# HIPAA Security Rule Changes RAG Assistant

This RAG application enables healthcare IT professionals to query in plain English the 2025 HIPAA Security Rule NPRM against the existing rule, receiving cited answers that identify what is changing and how to implement those changes.


In [ ]:
# Install dependencies (be patient--can take up to 90 seconds)
!pip install pymupdf sentence-transformers chromadb google-genai -q

print('Dependencies installed')

# Load necessary packages
import fitz #PyMuPDF - PDF parsing
from sentence_transformers import SentenceTransformer # Embedding model
import chromadb # Vector store
import google.genai as genai # Gemini API
import os # File path management
from google.colab import userdata # Secure API key access
import requests # PDF download from URLs
print('All imports successful')

In [ ]:
# Mount Google Drive for storage
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Initialize persistent ChromaDB client
chroma_client = chromadb.PersistentClient(
    path='/content/drive/MyDrive/hipaa_rag/chroma_db'
)

print('Drive mounted and ChromaDB initialized')

In [ ]:

# Define documents and URLs
DOCS = {
    'nprm_2025.pdf': 'https://www.govinfo.gov/content/pkg/FR-2025-01-06/pdf/2024-30983.pdf',
    'hipaa_security_rule_current.pdf': 'https://www.hhs.gov/sites/default/files/hipaa-simplification-201303.pdf'
}

# Create directory for storing document PDFs
PDF_DIR = '/content/drive/MyDrive/hipaa_rag/docs'
os.makedirs(PDF_DIR, exist_ok=True)


# Download PDFs from URLs
for filename, url in DOCS.items():
    filepath = os.path.join(PDF_DIR, filename)
    if os.path.exists(filepath):
      print(f'{filename} already exists -- skipping.')
    else:
      print(f'Downloading {filename}...')
      r = requests.get(url)
      if r.status_code != 200:
        print(f'Failed to download {filename}. Status code: {r.status_code}. Skipping ingestion for this file.')
        continue
      with open(filepath, 'wb') as f:
        f.write(r.content)
      print(f'Done.')


In [ ]:

# Initialize embedding model from HuggingFace
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Embedding model initialized')



In [ ]:
# Initialize Gemini client using the API key from Colab secrets
GOOGLE_API_KEY=userdata.get('Gemini') # Replace with name of your API key
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print('Gemini client initialized.')

In [ ]:
# Initialize ChromaDB collection
collection = chroma_client.get_or_create_collection(
    name='hipaa_docs',
    metadata={'hnsw:space': 'cosine'}
)

print('ChromaDB collection initialized')

In [10]:
def ingest_document(pdf_path, source_label):
  """
  Load a PDF, chunk it, embed chunks, and store in ChromaDB.
  source_label identifies which document chunks came from.
  """
  # Open PDF and extract text page by page
  doc = fitz.open(pdf_path)

  # Create empty lists
  chunks = []
  metadatas = []
  ids = []

  # Loop through individual pages of each doc, chunk by page
  for page_num, page in enumerate(doc):
    text=page.get_text() # Extract text from each page

    # Skip pages with very little text (headers, footers, blank pages)
    if len(text.strip()) < 100:
      continue

    chunks.append(text) # Append chunks to list
    metadatas.append({ # Create metadata for each chunk
        'source': source_label, # Source document for chunk
        'page': page_num + 1 # Page number of chunk within source document
    })
    ids.append(f'{source_label}_page_{page_num + 1}') # IDs to be used for citations in query response

  # Embed all chunks
  print(f'Embedding {len(chunks)} pages from {source_label}...')
  embeddings = embedding_model.encode(chunks).tolist() # ChromaDB expects list of vectors

  # Store in ChromaDB
  collection.add(
      documents=chunks,
      embeddings=embeddings,
      metadatas=metadatas,
      ids=ids
  )

  print(f'Done. {len(chunks)} chunks stored for {source_label}')

In [11]:
def retrieve_chunks(query, k=5):
  """
  Embed the query and retrieve the k most similar chunks from ChromaDB.
  Returns raw ChromaDB results including documents and metadata.
  """
  # Embed the query using same model as ingestion
  embeddings = embedding_model.encode(query) # Embed the query

  # Query ChromaDB for k nearest neighbors by cosin similarity
  results = collection.query(
    query_embeddings=[embeddings.tolist()],
    n_results=k
)
  # Return results to be passed to generate_response
  return results


In [12]:
def generate_response(query, results):
  """
  Build contgext from retrieved chunks and generated a grounded, cited response using Gemini.

  Args:
    query (str): The user's plain-English question
    results(dict): ChromaDB results from retrieve_chunks()

  Returns:
    str: A cited response grounded in the provided regulatory text.
  """

  # Define system prompt for response
  system_prompt = """
    -You are a helpful healthcare privacy assistant with legal knowledge.
    -You are helping someone with knowledge of healthcare privacy and healthcare IT, but not strong legal knowledge.
    -You are helping an organization that is compliant with current HIPAA regulations but is unsure of how the proposed changes will mean for them.

    Core Rules:
    -Answer questions using only the provided context and cite your sources in responses.
    -Never provide incomplete answers or hallucinate answers.
    -Do not provide answers that are not supported by the provided context.
    -Where appropriate, identify specific changes from the current rule to the proposed new rule.
    -Only answer questions that can be answered with the context.
    -If the question cannot be answered with the context, respond "I am not allowed to answer that question."
    -Your response may include directions to implement the specific proposed changes.
    -All responses with directions must help accomplish a specific goal in the proposed changes.

    Output Rules:
    -Responses should include a brief summary (1-2 sentences) responding to the question.
    -Responses should identify actionable steps for implementing proposed changes.
    -If the query does not require actionable steps, end your response with "No actions are required."
    """
  # Extract documents and metadata from reults
  documents = results['documents'][0]
  metadata_list = results['metadatas'][0]
  # Parse documents and metadata_list results into context to be used to respond to query
  context = ""
  for document, metadata in zip(documents, metadata_list):
      context += f'[Source: {metadata['source']}, Page: {metadata['page']}]\n{document}\n\n'

  # Pass context and query to Gemini and return cited response
  response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    config=genai.types.GenerateContentConfig(
        system_instruction=system_prompt
    ),
    contents=f'Context:\n{context}\n\nQuestion: {query}' # Format message to LLM to define context and query
  )

  return response.text # Return text of LLM response

In [13]:
# Enter your query here
query = 'What are three most significant changes with the new rule?'
results = retrieve_chunks(query)
answer = generate_response(query, results)
print(answer)

Based on the proposed changes, three significant shifts from the current HIPAA Security Rule include:

1.  **Elevation and Increased Specificity of the Risk Analysis Requirement**: The requirement to conduct a risk analysis is elevated from an implementation specification to a standard (proposed 45 CFR 164.308(a)(2)(i)). This new standard mandates an "accurate and comprehensive written assessment" with eight specific documentation requirements (e.g., inventorying technology assets, identifying threats and vulnerabilities, assessing security measures, determining likelihood and impact, and assessing risk levels). It also requires reviewing, verifying, and updating this assessment at least every 12 months or in response to environmental/operational changes (NPRM, Page 44). Furthermore, evaluations of potential changes must now be in writing and performed *before* a change is made to assess its impact on ePHI (NPRM, Page 44).
2.  **Mandatory Implementation of Encryption and Removal of Add